# DS2002 · Capstone Kickoff and First Pulls

**Studio — 2026-11-18 · Fall 2026**  
**Class time:** 45 minutes

---

## Capstone kickoff and first pulls

The capstone starts today. Read `DS2002 Capstone — ACC Reality Check.pdf` in this folder before Friday, because it describes everything your team will hand in and when. The short version is that your team will decide which ACC men's basketball teams are as good as their records say and which ones have built their records on easy opponents. To do that you will combine a ratings file with game results from a live API, keep everything in a SQLite database, answer the questions with SQL, and publish your results as a small web API.

Today has a modest goal. Each team forms, sets up a repository, and looks at both sources with its own eyes. You will not clean or load anything yet. The point of the hour is to find out what the data looks like before anyone writes a plan around it.

If you do not follow basketball, start with the appendix of the PDF. It explains every term used in this notebook, and nobody on your team should feel behind because of the sport.

### Step 1: Form your team

Teams have three or four people. Fill in the cell below together and commit it to the team repository by the end of class. The roles are a starting point, and you can trade them later, but someone should own each part of the project from the first day.

In [1]:
team = {
    'name': 'MNM',
    'members': ['Nathaniel Garcia', 'Mouad Tazka', 'Mohamed Hassan'],
    'repo_url': 'https://github.com/NateG-10/ds-2002-final-project',
    'roles': {
        'ratings CSV and crosswalk': 'Nathaniel Garcia',
        'ESPN pulls and box scores': 'Mouad Tazka',
        'SQLite schema and queries': 'Mohamed Hassan',
        'API and PythonAnywhere': 'Mohamed Hassan',
    },
    'questions': {
        'Q1 and Q2': 'Nathaniel Garcia',
        'Q4 and Q5': 'Mouad Tazka',
        'Q3 and Q6': 'Mohamed Hassan',
    },
    'meeting_time': 'As needed, but at least once a week',
}

unfilled = [k for k, v in team.items() if 'TODO' in str(v)]
print('still unfilled:', unfilled if unfilled else 'nothing, the charter is complete')

still unfilled: nothing, the charter is complete


### Step 2: Pull the ratings CSV

Bart Torvik publishes a rating for every Division I team as a CSV file at this address:

`https://barttorvik.com/2027_team_results.csv`

The 2027 in the name refers to the season that ends in 2027, which is the season being played right now. You do not need a key or an account.

**TODO:** download the file with `requests`, read it into pandas, and look at it before you do anything else. Print the shape, the column names, and the first few rows. Save the raw text to a file named with today's date, because the ratings change every day and you will want to know which version you used.

In [2]:
%pip install pandas
from pathlib import Path
import csv
import json
import requests
import pandas as pd

# Fixed date identifies this saved kickoff pull; use a NEW date for future snapshots.
PULL_DATE = '2026-11-18'
RAW = Path('raw')
RAW.mkdir(exist_ok=True)
TORVIK_URL = 'https://barttorvik.com/2027_team_results.csv'

def saved_response(url, path, params=None):
    if path.exists():
        print('Reading saved response:', path)
    else:
        response = requests.get(url, params=params, timeout=45)
        response.raise_for_status()
        path.write_bytes(response.content)
        print('Downloaded:', response.url, 'HTTP', response.status_code)
    return path

ratings_path = saved_response(TORVIK_URL, RAW / f'torvik_{PULL_DATE}.csv')
ratings = pd.read_csv(ratings_path)
print('Shape:', ratings.shape)
print('Columns:', ratings.columns.tolist())
display(ratings.head())
acc = ratings.loc[ratings['conf'].eq('ACC')]
print('ACC rows:', len(acc))
display(acc[['team', 'conf', 'record', 'rank', 'rank.1', 'barthag']])
with ratings_path.open(newline='') as f:
    original_headers = next(csv.reader(f))
print('Original headers named rank:', original_headers.count('rank'))
print('Rank columns equal in this snapshot:', ratings['rank'].equals(ratings['rank.1']))
print('Barthag decreases as rank.1 increases:',
      ratings.sort_values('rank.1')['barthag'].is_monotonic_decreasing)


Note: you may need to restart the kernel to use updated packages.
Reading saved response: raw/torvik_2026-11-18.csv
Shape: (365, 45)
Columns: ['rank', 'team', 'conf', 'record', 'adjoe', 'oe Rank', 'adjde', 'de Rank', 'barthag', 'rank.1', 'proj. W', 'Proj. L', 'Pro Con W', 'Pro Con L', 'Con Rec.', 'sos', 'ncsos', 'consos', 'Proj. SOS', 'Proj. Noncon SOS', 'Proj. Con SOS', 'elite SOS', 'elite noncon SOS', 'Opp OE', 'Opp DE', 'Opp Proj. OE', 'Opp Proj DE', 'Con Adj OE', 'Con Adj DE', 'Qual O', 'Qual D', 'Qual Barthag', 'Qual Games', 'FUN', 'ConPF', 'ConPA', 'ConPoss', 'ConOE', 'ConDE', 'ConSOSRemain', 'Conf Win%', 'WAB', 'WAB Rk', 'Fun Rk', 'adjt']


,rank,team,conf,record,adjoe,oe Rank,adjde,de Rank,barthag,rank.1,...,ConPA,ConPoss,ConOE,ConDE,ConSOSRemain,Conf Win%,WAB,WAB Rk,Fun Rk,adjt
0,1,Kansas,B12,5-0,122.210929,6,92.797460,3,0.959546,1,...,0.0,0.0,0,0,0.843819,0,1.603877,2,80,72.550470
1,2,Houston,B12,5-0,122.416439,5,93.279053,5,0.957956,2,...,0.0,0.0,0,0,0.847790,0,1.592145,3,59,67.258776
2,3,Duke,ACC,3-2,119.880679,14,92.060397,1,0.954200,3,...,0.0,0.0,0,0,0.805033,0,-0.442862,120,353,71.811124
3,4,Illinois,B10,4-1,124.777422,1,96.411380,19,0.951011,4,...,0.0,0.0,0,0,0.839177,0,0.981213,10,116,74.254181
4,5,Florida,SEC,5-1,123.133092,2,95.403194,11,0.949515,5,...,0.0,0.0,0,0,0.858506,0,1.732562,1,100,69.545494


ACC rows: 18


,team,conf,record,rank,rank.1,barthag
2,Duke,ACC,3-2,3,3,0.954200
5,Virginia,ACC,6-0,6,6,0.944878
15,Miami FL,ACC,4-1,16,16,0.920269
17,Louisville,ACC,2-1,18,18,0.919139
31,N.C. State,ACC,4-0,32,32,0.881941
36,Virginia Tech,ACC,5-0,37,37,0.873199
48,North Carolina,ACC,3-1,49,49,0.847129
52,Florida St.,ACC,4-1,53,53,0.834588
60,SMU,ACC,5-0,61,61,0.822848
67,Notre Dame,ACC,5-1,68,68,0.811654


Original headers named rank: 2
Rank columns equal in this snapshot: True
Barthag decreases as rank.1 increases: True


**TODO:** answer these in the cell below, using what you printed.

1. How many rows are there, and how many belong to the ACC? You should be able to filter on one column to find out.
2. Two of the original headers have the same name. What did pandas call them, and which one would you trust as the team's rank?
3. Look at how a few schools are spelled, such as North Carolina State and Miami. Write the spellings down exactly, because the other source will spell them differently.

1. The saved November 18 pull has **365 rows and 45 columns**. Filtering `conf == 'ACC'` returns **18 rows**.
2. The raw header contains `rank` twice; pandas calls these `rank` and `rank.1`. I would use **`rank.1`** for the national Barthag rank because Appendix B defines it that way. The printed checks show both columns are equal in this snapshot and Barthag decreases as `rank.1` increases; equality today alone does not establish their meanings.
3. The exact Torvik names include **`N.C. State`**, **`Miami FL`**, and **`Florida St.`**. Preserve these source spellings when building the later crosswalk.

### Step 3: Pull one day of games from ESPN

ESPN serves game results as JSON with no key. This address returns every game an ACC team played on one date:

`https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard`

Pass the parameters `groups=2` (the ACC), `dates=YYYYMMDD`, and `limit=200`. Pick a date from the past week. You practiced this kind of request in October, so reuse what worked for you then, including a timeout and a check of the status code.

**TODO:** make the request, then explore the JSON one level at a time. Find the list of games, count them, and for one game print both team names, both scores, whether the game is finished, and the start time exactly as ESPN wrote it.

In [3]:
ESPN_URL = 'https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard'
GAME_DATE = '20261114'  # Within the week before the November 18 kickoff.
scoreboard_path = saved_response(
    ESPN_URL, RAW / f'espn_scoreboard_{GAME_DATE}.json',
    params={'groups': 2, 'dates': GAME_DATE, 'limit': 200})
scoreboard = json.loads(scoreboard_path.read_text())
print('Top-level keys:', list(scoreboard))
events = scoreboard['events']
print('Games:', len(events))
if events:
    print('First event keys:', list(events[0]))
    print('First competition keys:', list(events[0]['competitions'][0]))
for event in events:
    competition = event['competitions'][0]
    print('\nGame ID:', event['id'])
    print('Start exactly as ESPN wrote it:', competition['date'])
    print('Eastern time:', pd.Timestamp(competition['date']).tz_convert('America/New_York'))
    print('Finished:', competition['status']['type']['completed'])
    print('Neutral site:', competition['neutralSite'])
    for side in competition['competitors']:
        print(side['homeAway'], side['team']['location'],
              '| full name:', side['team']['displayName'],
              '| score:', repr(side['score']), '| type:', type(side['score']).__name__)

# Supplemental directory: NC State, Miami and Florida State did not play on this date.
TEAMS_URL = 'https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/teams'
teams_path = saved_response(TEAMS_URL, RAW / f'espn_teams_{PULL_DATE}.json', {'limit': 1000})
directory = json.loads(teams_path.read_text())
print('\nSupplemental ESPN directory spellings (not games on November 14):')
for entry in directory['sports'][0]['leagues'][0]['teams']:
    team_info = entry['team']
    if team_info['id'] in {'152', '2390', '52'}:
        print(team_info['id'], repr(team_info['location']), repr(team_info['displayName']))
print('\nOpponent rows in Torvik (inspection only):')
display(ratings.loc[ratings['team'].isin(['UNC Asheville', 'Southern Utah', 'Utah Tech']),
                    ['team', 'conf', 'rank.1']])


Reading saved response: raw/espn_scoreboard_20261114.json
Top-level keys: ['leagues', 'groups', 'events', 'provider', 'eventsDate']
Games: 3
First event keys: ['id', 'uid', 'date', 'name', 'shortName', 'season', 'competitions', 'links', 'status']
First competition keys: ['id', 'uid', 'date', 'attendance', 'type', 'timeValid', 'neutralSite', 'conferenceCompetition', 'playByPlayAvailable', 'recent', 'venue', 'competitors', 'notes', 'status', 'broadcasts', 'format', 'startDate', 'broadcast', 'geoBroadcasts', 'highlights', 'headlines']

Game ID: 401920307
Start exactly as ESPN wrote it: 2026-11-14T21:00Z
Eastern time: 2026-11-14 16:00:00-05:00
Finished: True
Neutral site: False
home Georgia Tech | full name: Georgia Tech Yellow Jackets | score: '75' | type: str
away UNC Asheville | full name: UNC Asheville Bulldogs | score: '74' | type: str

Game ID: 401909728
Start exactly as ESPN wrote it: 2026-11-14T21:00Z
Eastern time: 2026-11-14 16:00:00-05:00
Finished: True
Neutral site: False
home W

,team,conf,rank.1
257,Southern Utah,BSky,258
294,Utah Tech,BSky,295
296,UNC Asheville,BSth,297


**TODO:** answer these in the cell below.

1. Where in the JSON are the games, and where inside one game are the two teams?
2. Compare the start time ESPN gives for an evening game with the date you asked for. What time zone is the start time in, and what goes wrong if you use its first ten characters as the game date?
3. How does ESPN spell the schools you wrote down in Step 2?
4. Did every opponent on your date play in Division I? Look the unfamiliar names up.

1. Games are in `scoreboard['events']`; the two teams are in `events[i]['competitions'][0]['competitors']`. Their short names are at `['team']['location']`, and scores are strings at `['score']`. The November 14 response contains **3 games**, all completed: Georgia Tech 75–74 UNC Asheville, Wake Forest 89–51 Southern Utah, and California 79–73 Utah Tech.
2. ESPN supplies **UTC**, indicated by `Z`. Our actual starts are `2026-11-14T21:00Z` (4 pm Eastern, two games) and `2026-11-14T22:00Z` (5 pm Eastern, one game). **None of these three crosses the UTC date boundary**, so this date does not demonstrate the evening rollover. Appendix D's example, `2026-11-15T00:00Z`, is 7 pm Eastern on November 14; taking its first ten characters would incorrectly assign November 15. Convert with `tz_convert('America/New_York')` before extracting the date. That example comes from the PDF, not this response.
3. The separately saved ESPN team-directory output verifies Torvik `N.C. State` → ESPN `NC State`, `Miami FL` → `Miami`, and `Florida St.` → `Florida State`. ESPN full display names additionally include mascots (`NC State Wolfpack`, `Miami Hurricanes`, `Florida State Seminoles`). None of these schools appears in this day's three games.
4. **Yes, all three opponents are Division I** and all three have rows in our Torvik snapshot. The unfamiliar schools were also checked against official sources: [UNC Asheville / NCAA directory](https://web3.ncaa.org/directory/orgDetail?id=456), [Southern Utah Athletics](https://suutbirds.com/news/2026/7/1/a-new-era-begins-southern-utah-athletics-enters-a-new-chapter.aspx), and [Utah Tech Athletics](https://utahtechtrailblazers.com/news/2024/8/1/general-its-official-utah-tech-university-completes-ncaa-division-i-transition.aspx). This conclusion covers this date only; an unmatched name on another date would require investigation, not an automatic non-Division-I label.

### Step 4: Write down the work ahead

You have now seen both sources. Before you leave, list at least five specific things that will have to be cleaned, converted, or matched before the two sources can be joined, and make each one concrete. "Names are different" is less useful than "ESPN writes NC State and Torvik writes N.C. State." This list becomes the first entries in your decision log on Friday.

These are tasks for the next stage; no cleaned tables or database loads have been created today.

1. Build explicit source-name mappings: `N.C. State` ↔ `NC State`, `Miami FL` ↔ `Miami`, and `Florida St.` ↔ `Florida State`; keep ESPN IDs as stable identifiers and validate all 18 ACC teams without duplicate matches.
2. Select `rank.1` as the documented Barthag rank; preserve the original duplicate `rank` headers in raw files and use unambiguous names in the later database.
3. Convert score strings such as Georgia Tech's `'75'` and UNC Asheville's `'74'` to integers before calculating a margin of 1; flag missing or invalid scores rather than treating them as zero.
4. Convert UTC starts to `America/New_York` before deriving game dates. Today's three starts stay on November 14, but the PDF's midnight-UTC example belongs to the previous Eastern date.
5. Check `competitions[0]['status']['type']['completed']` before keeping results. All 3 games here pass; later requests may include unfinished games.
6. Flatten each competition's two competitors into two team-game rows (these 3 games would become 6 rows). Preserve `homeAway` and `neutralSite`; do not assume the first competitor is always the home team.
7. Investigate unmatched opponents before assigning missing ratings. All 3 opponents here have Torvik rows; future genuinely unrated opponents need a documented policy and affected-row counts.
8. Keep the snapshot date (`2026-11-18`) separate from the game date (`2026-11-14`). This snapshot is after these games and cannot be used as a pregame prediction for Q6.
9. Preserve Torvik records such as Duke's `3-2` as raw text and split wins/losses into numeric fields only when needed. Later compare their coverage with the extracted finished-game counts.

Box-score shooting splits and player flattening are additional final-project requirements, but no summary endpoint was pulled today, so they are not claimed as observations from this kickoff.

### Before Friday

Friday's check-in notebook walks your team through getting the first version of the pipeline running, including the name crosswalk and the first tables in SQLite. Read the PDF tonight, and agree as a team on who is starting which part before you leave the room.

### Team progress

This cell is for your team, and nothing is submitted or graded. The only graded work in the capstone is the final submission on Friday, December 18, so these progress cells exist to help you notice early when something is behind. Fill it in and commit it with the rest of the notebook.

In [4]:
progress = {
    'acc_rows_in_torvik': len(acc),
    'espn_date_pulled': GAME_DATE,
    'espn_games_that_day': len(events),
    'name_that_differs': 'Torvik: N.C. State; ESPN location: NC State (team directory)',
    'blocker': 'Awaiting team name and meeting time',
}
for k, v in progress.items():
    print(f'{k}: {v}')


acc_rows_in_torvik: 18
espn_date_pulled: 20261114
espn_games_that_day: 3
name_that_differs: Torvik: N.C. State; ESPN location: NC State (team directory)
blocker: Awaiting team name and meeting time
